<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/Fabian_Timing_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimized Fabian Timing Model Backtest

## Strategy Name
Optimized Fabian Timing Model (Classical Richard Fabian Framework adapted for modern market regimes).

## Core Objective
Test an intermarket trend-following confirmation matrix across three major asset classes to dynamically filter market regimes and protect capital from systemic bear markets.

## Asset Tickers Used
*   **SPY**: S&P 500 Index ETF - Core Equity Vector
*   **DIA**: Dow Jones Industrial Average ETF - Industrial Breadth Vector
*   **XLU**: Utilities Sector ETF - Macro Rate/Liquidity Canary

## Core Logic
Evaluated on strict weekly intervals (Friday 3:45 PM EST intraday evaluation). A 100% Long SPY signal is triggered only under unanimous confirmation where all 3 assets close above their respective $N$-week Simple Moving Average (SMA). The strategy shifts to 100% Cash if two or more assets violate their moving averages.

## Key Modification
The classical trailing drop filter was tested and discarded to eliminate transactional friction and whipsaws during standard mid-cycle pullbacks.

## Current Testing Focus
Evaluating a multi-period parameter sweep across lookback windows ranging from 10 to 90 weeks to identify the optimal balance between trailing drawdowns, absolute dollar returns, and risk-adjusted efficiency (Sharpe Ratio) in post-2000 market structures.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

In [18]:
# ==========================================
# 1. PARAMETERS & INPUTS
# ==========================================
sma_period = 39         # N-week moving average period
trailing_drop = 8.0     # X% allowable drop from peak before manual exit
start_date = "2000-01-01"
tickers = ["SPY", "DIA", "XLU"]

print(f"Initializing Fabian Timing Model...")
print(f"Parameters -> SMA: {sma_period} weeks | Trailing Drop: {trailing_drop}%")

# ==========================================
# 2. DATA INGESTION & FRIDAY INTRADAY PATCH
# ==========================================
raw_data = yf.download(tickers, start=start_date, progress=False)['Close']

# Check if today is Friday and it's near/after 3:45 PM EST
now = datetime.now()
if now.weekday() == 4:
    print("Detected Friday execution. Fetching latest intraday snapshot for 3:45 PM evaluation...")
    today_intraday = yf.download(tickers, period="1d", interval="1m", progress=False)['Close']
    if not today_intraday.empty:
        latest_snapshot = today_intraday.iloc[-1]
        today_date = pd.Timestamp(now.date())
        raw_data.loc[today_date] = latest_snapshot
        print(f"--> Patched live Friday prices into analysis matrix: {dict(latest_snapshot.round(2))}")

weekly = raw_data.resample('W-FRI').last().dropna()

# ==========================================
# 3. ALGORITHM ENGINE
# ==========================================
for ticker in tickers:
    weekly[f'{ticker}_SMA'] = weekly[ticker].rolling(window=sma_period).mean()

weekly = weekly.dropna()

for ticker in tickers:
    weekly[f'{ticker}_Above'] = weekly[ticker] > weekly[f'{ticker}_SMA']

weekly['Regime_Signal'] = (weekly[[f'{t}_Above' for t in tickers]].sum(axis=1) == 3).astype(int)

signals = []
in_position = False
peak_price = 0.0

for idx, row in weekly.iterrows():
    spy_price = row['SPY']
    regime = row['Regime_Signal']

    if not in_position:
        if regime == 1:
            in_position = True
            peak_price = spy_price
            signals.append(1)
        else:
            signals.append(0)
    else:
        if spy_price > peak_price:
            peak_price = spy_price

        stop_level = peak_price * (1 - (trailing_drop / 100.0))

        if spy_price <= stop_level:
            in_position = False
            signals.append(0)
        elif regime == 0:
            in_position = False
            signals.append(0)
        else:
            signals.append(1)

weekly['Final_Signal'] = signals
weekly['Executed_Position'] = weekly['Final_Signal'].shift(1).fillna(0)

# ==========================================
# 4. PERFORMANCE & RISK METRICS
# ==========================================
# Calculate baseline returns
weekly['SPY_Returns'] = np.log(weekly['SPY'] / weekly['SPY'].shift(1))
weekly['Strategy_Returns'] = weekly['SPY_Returns'] * weekly['Executed_Position']

# Cumulative growth series
weekly['Cum_BH'] = np.exp(weekly['SPY_Returns'].cumsum()) * 100000
weekly['Cum_Strategy'] = np.exp(weekly['Strategy_Returns'].cumsum()) * 100000

# Annualized Sharpe Ratio Calculations (Annualization factor = 52 for weekly data)
# Simple daily/weekly returns work best here rather than log returns for standard metrics
spy_simple = weekly['SPY'].pct_change()
strat_simple = spy_simple * weekly['Executed_Position']

sharpe_bh = (spy_simple.mean() / spy_simple.std()) * np.sqrt(52) if spy_simple.std() != 0 else 0
sharpe_strat = (strat_simple.mean() / strat_simple.std()) * np.sqrt(52) if strat_simple.std() != 0 else 0

# Maximum Drawdown Calculations
# Buy & Hold Drawdown
peak_bh = weekly['Cum_BH'].cummax()
dd_bh = (weekly['Cum_BH'] - peak_bh) / peak_bh
max_dd_bh = dd_bh.min() * 100

# Strategy Drawdown
peak_strat = weekly['Cum_Strategy'].cummax()
dd_strat = (weekly['Cum_Strategy'] - peak_strat) / peak_strat
max_dd_strat = dd_strat.min() * 100

print("\n==============================================")
print("             BACKTEST METRICS                 ")
print("==============================================")
print(f"Final Value Buy & Hold (SPY): ${weekly['Cum_BH'].iloc[-1]:,.2f}")
print(f"Final Value Fabian Strategy:  ${weekly['Cum_Strategy'].iloc[-1]:,.2f}")
print(f"Time Spent Invested:          {weekly['Executed_Position'].mean() * 100:.1f}%")
print("----------------------------------------------")
print(f"Sharpe Ratio (Buy & Hold):    {sharpe_bh:.3f}")
print(f"Sharpe Ratio (Strategy):      {sharpe_strat:.3f}")
print("----------------------------------------------")
print(f"Max Drawdown (Buy & Hold):    {max_dd_bh:.2f}%")
print(f"Max Drawdown (Strategy):      {max_dd_strat:.2f}%")
print("==============================================")

Initializing Fabian Timing Model...
Parameters -> SMA: 48 weeks | Trailing Drop: 8.0%


/tmp/ipykernel_887/233459418.py:15: FutureWarning:

YF.download() has changed argument auto_adjust default to True




             BACKTEST METRICS                 
Final Value Buy & Hold (SPY): $894,698.02
Final Value Fabian Strategy:  $461,273.27
Time Spent Invested:          62.9%
----------------------------------------------
Sharpe Ratio (Buy & Hold):    0.576
Sharpe Ratio (Strategy):      0.659
----------------------------------------------
Max Drawdown (Buy & Hold):    -54.61%
Max Drawdown (Strategy):      -17.47%


## Range Testing Section (Parameter Sweep)

This section performs a comprehensive parameter sweep to identify optimal `SMA_Period` values for the Fabian Timing Model. It evaluates strategy performance across a range of Simple Moving Average lookback windows, focusing on absolute returns, risk-adjusted returns (Sharpe Ratio), and maximum drawdown.

In [19]:
# ==========================================
# 5. INTERACTIVE PLOTLY VISUALIZATION
# ==========================================
fig = go.Figure()

# Add Buy & Hold Line
fig.add_trace(go.Scatter(
    x=weekly.index,
    y=weekly['Cum_BH'],
    mode='lines',
    name='Buy & Hold (SPY)',
    line=dict(color='darkgray', width=1.5),
    hovertemplate='<b>Buy & Hold</b><br>Date: %{x}<br>Value: $%{y:,.2f}<extra></extra>'
))

# Add Fabian Strategy Line
fig.add_trace(go.Scatter(
    x=weekly.index,
    y=weekly['Cum_Strategy'],
    mode='lines',
    name=f'Fabian Model (SMA: {sma_period}, Drop: {trailing_drop}%)',
    line=dict(color='#1f77b4', width=2.5),
    hovertemplate='<b>Fabian Strategy</b><br>Date: %{x}<br>Value: $%{y:,.2f}<br>Position: %{customdata}<extra></extra>',
    customdata=weekly['Executed_Position'].map({1: "100% Long SPY", 0: "100% Cash/Flat"})
))

# Layout configurations for crisp tracking
fig.update_layout(
    title=dict(
        text=f"Fabian Timing Model Portfolio Comparison<br><sup>SMA Period: {sma_period} Weeks | Trailing Drop Limit: {trailing_drop}%</sup>",
        font=dict(size=18, family="Arial")
    ),
    xaxis=dict(
        title="Timeline",
        showgrid=True,
        gridcolor='rgba(200, 200, 200, 0.2)',
        rangeslider=dict(visible=True)  # Adds a bottom timeline range selector
    ),
    yaxis=dict(
        title="Portfolio Balance ($ Base 100k)",
        type="log",                    # Log scale cleanly balances 20+ year compounding
        showgrid=True,
        gridcolor='rgba(200, 200, 200, 0.2)',
        tickformat="$,"
    ),
    template="plotly_white",
    hovermode="x unified",               # Combines hover labels over a vertical baseline
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255, 255, 255, 0.8)")
)

# Render inside Google Colab cell
fig.show()

In [14]:
# ==========================================
# 1. PARAMETERS & CONFIGURATION
# ==========================================
start_date = "2000-01-01"
tickers = ["SPY", "DIA", "XLU"]

# Define the range of SMA periods you want to test (e.g., from 10 weeks to 50 weeks)
sma_range = range(10, 91)

print(f"Initializing Multi-Period Fabian Sweep (No Trailing Drop Filter)...")
print(f"Testing SMA Range: {min(sma_range)} to {max(sma_range)} weeks\n")

# ==========================================
# 2. DATA INGESTION & FRIDAY INTRADAY PATCH
# ==========================================
raw_data = yf.download(tickers, start=start_date, progress=False)['Close']

# Check if today is Friday and it's near/after 3:45 PM EST
now = datetime.now()
if now.weekday() == 4:
    today_intraday = yf.download(tickers, period="1d", interval="1m", progress=False)['Close']
    if not today_intraday.empty:
        latest_snapshot = today_intraday.iloc[-1]
        today_date = pd.Timestamp(now.date())
        raw_data.loc[today_date] = latest_snapshot

weekly_base = raw_data.resample('W-FRI').last().dropna()

# ==========================================
# 3. OPTIMIZATION SWEEP ENGINE
# ==========================================
results_list = []

# Baseline Buy & Hold Calculations (Only need to compute this once)
spy_simple = weekly_base['SPY'].pct_change()
cum_bh = np.exp(np.log(weekly_base['SPY'] / weekly_base['SPY'].shift(1)).cumsum()) * 100000
final_bh = cum_bh.iloc[-1]
sharpe_bh = (spy_simple.mean() / spy_simple.std()) * np.sqrt(52) if spy_simple.std() != 0 else 0
peak_bh = cum_bh.cummax()
max_dd_bh = ((cum_bh - peak_bh) / peak_bh).min() * 100

# Loop through each individual SMA period
for period in sma_range:
    df = weekly_base.copy()

    # Calculate SMAs for this specific period
    for ticker in tickers:
        df[f'{ticker}_SMA'] = df[ticker].rolling(window=period).mean()
    df = df.dropna()

    if df.empty:
        continue

    # Generate Regime Signals (1 if all 3 indices are above SMA, else 0)
    for ticker in tickers:
        df[f'{ticker}_Above'] = df[ticker] > df[f'{ticker}_SMA']

    df['Regime_Signal'] = (df[[f'{t}_Above' for t in tickers]].sum(axis=1) == 3).astype(int)

    # Shift signal by 1 week for execution tracking
    df['Executed_Position'] = df['Regime_Signal'].shift(1).fillna(0)

    # Calculate Strategy Performance Metrics
    df['SPY_Returns'] = np.log(df['SPY'] / df['SPY'].shift(1))
    df['Strategy_Returns'] = df['SPY_Returns'] * df['Executed_Position']
    df['Cum_Strategy'] = np.exp(df['Strategy_Returns'].cumsum()) * 100000

    # Risk Metrics
    spy_pct = df['SPY'].pct_change()
    strat_pct = spy_pct * df['Executed_Position']

    sharpe_strat = (strat_pct.mean() / strat_pct.std()) * np.sqrt(52) if strat_pct.std() != 0 else 0

    peak_strat = df['Cum_Strategy'].cummax()
    max_dd_strat = ((df['Cum_Strategy'] - peak_strat) / peak_strat).min() * 100
    time_invested = df['Executed_Position'].mean() * 100

    # Store data for comparison
    results_list.append({
        'SMA_Period': period,
        'Final_Value': df['Cum_Strategy'].iloc[-1],
        'Sharpe_Ratio': sharpe_strat,
        'Max_Drawdown': max_dd_strat,
        'Time_In_Market': time_invested
    })

# Convert results matrix into a structured DataFrame
results_df = pd.DataFrame(results_list)

# ==========================================
# 4. REPORTING REPORT
# ==========================================
# Sort to locate the highest absolute returning period
best_return = results_df.sort_values(by='Final_Value', ascending=False).iloc[0]
# Sort to locate the optimal risk-adjusted period
best_sharpe = results_df.sort_values(by='Sharpe_Ratio', ascending=False).iloc[0]

print("==========================================================")
print("             BASELINE BENCHMARK PERFORMANCE               ")
print("==========================================================")
print(f"Buy & Hold Final Value:  ${final_bh:,.2f}")
print(f"Buy & Hold Sharpe Ratio: {sharpe_bh:.3f}")
print(f"Buy & Hold Max Drawdown: {max_dd_bh:.2f}%")
print("==========================================================\n")

print("==========================================================")
print("             OPTIMIZATION SWEEP RESULTS                   ")
print("==========================================================")
print(f"Top Absolute Return:  {int(best_return['SMA_Period'])}-week SMA")
print(f"  -> Final Value:     ${best_return['Final_Value']:,.2f}")
print(f"  -> Sharpe Ratio:    {best_return['Sharpe_Ratio']:.3f}")
print(f"  -> Max Drawdown:    {best_return['Max_Drawdown']:.2f}%")
print(f"  -> Market Exposure: {best_return['Time_In_Market']:.1f}%")
print("----------------------------------------------------------")
print(f"Top Sharpe Ratio:     {int(best_sharpe['SMA_Period'])}-week SMA")
print(f"  -> Final Value:     ${best_sharpe['Final_Value']:,.2f}")
print(f"  -> Sharpe Ratio:    {best_sharpe['Sharpe_Ratio']:.3f}")
print(f"  -> Max Drawdown:    {best_sharpe['Max_Drawdown']:.2f}%")
print(f"  -> Market Exposure: {best_sharpe['Time_In_Market']:.1f}%")
print("==========================================================")

Initializing Multi-Period Fabian Sweep (No Trailing Drop Filter)...
Testing SMA Range: 10 to 90 weeks



/tmp/ipykernel_887/962810948.py:16: FutureWarning:

YF.download() has changed argument auto_adjust default to True



             BASELINE BENCHMARK PERFORMANCE               
Buy & Hold Final Value:  $816,961.40
Buy & Hold Sharpe Ratio: 0.535
Buy & Hold Max Drawdown: -54.61%

             OPTIMIZATION SWEEP RESULTS                   
Top Absolute Return:  78-week SMA
  -> Final Value:     $565,777.37
  -> Sharpe Ratio:    0.671
  -> Max Drawdown:    -24.38%
  -> Market Exposure: 71.1%
----------------------------------------------------------
Top Sharpe Ratio:     48-week SMA
  -> Final Value:     $489,955.71
  -> Sharpe Ratio:    0.681
  -> Max Drawdown:    -17.47%
  -> Market Exposure: 63.1%


In [15]:
# Ensure your previous cell has successfully generated 'results_df'
# Check if the results dataframe is present before trying to plot
if 'results_df' in locals() and not results_df.empty:

    # Sort results sequentially by SMA period for linear line continuity
    plot_df = results_df.sort_values('SMA_Period')

    # ==================================================================
    # CHART 1: DUAL-AXIS LINE METRIC CHART (RETURNS VS SHARPE)
    # ==================================================================
    fig_metrics = make_subplots(specs=[[{"secondary_y": True}]])

    # Line trace for Absolute Final Value
    fig_metrics.add_trace(
        go.Scatter(
            x=plot_df['SMA_Period'],
            y=plot_df['Final_Value'],
            mode='lines+markers',
            name='Final Portfolio Value ($)',
            line=dict(color='#1f77b4', width=2.5),
            marker=dict(size=6),
            hovertemplate='<b>%{x}-Week SMA</b><br>Portfolio Value: $%{y:,.2f}<extra></extra>'
        ),
        secondary_y=False
    )

    # Line trace for Sharpe Ratio
    fig_metrics.add_trace(
        go.Scatter(
            x=plot_df['SMA_Period'],
            y=plot_df['Sharpe_Ratio'],
            mode='lines+markers',
            name='Sharpe Ratio',
            line=dict(color='#ff7f0e', width=2, dash='dot'),
            marker=dict(size=6, symbol='square'),
            hovertemplate='<b>%{x}-Week SMA</b><br>Sharpe Ratio: %{y:.3f}<extra></extra>'
        ),
        secondary_y=True
    )

    fig_metrics.update_layout(
        title=dict(text="<b>Fabian Parameters vs Portfolio Returns & Efficiency</b>", font=dict(size=16)),
        xaxis=dict(title="SMA Lookback Period (Weeks)", showgrid=True, gridcolor='rgba(200,200,200,0.2)'),
        yaxis=dict(title="Final Portfolio Balance ($)", tickformat="$,", showgrid=True, gridcolor='rgba(200,200,200,0.2)'),
        yaxis2=dict(title="Annualized Sharpe Ratio", showgrid=False),
        template="plotly_white",
        hovermode="x unified",
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)")
    )

    # ==================================================================
    # CHART 2: DRAWDOWN MATRIX INTENSITY HEATMAP (1D)
    # ==================================================================
    # We construct a 1D grid layout by feeding the 1D matrix into an array wrapper
    z_drawdowns = [plot_df['Max_Drawdown'].values]
    x_weeks = [f"{w}w" for w in plot_df['SMA_Period']]

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=z_drawdowns,
        x=plot_df['SMA_Period'],
        y=['Max Drawdown (%)'],
        colorscale='Reds_r',  # Reverses color scaling so that deepest drops stand out in heavy dark crimson
        colorbar=dict(title="Max DD %", tickformat=".1f%"),
        hovertemplate='<b>%{x}-Week SMA</b><br>Peak-to-Trough Drawdown: %{z:.2f}%<extra></extra>'
    ))

    fig_heatmap.update_layout(
        title=dict(text="<b>Drawdown Penalty Matrix Across Lookback Periods</b>", font=dict(size=16)),
        xaxis=dict(title="SMA Lookback Period (Weeks)", dtick=2, showgrid=False),
        yaxis=dict(fixedrange=True, showgrid=False),
        height=250,
        template="plotly_white"
    )

    # Display both panels consecutively in the notebook window
    fig_metrics.show()
    fig_heatmap.show()

else:
    print("Error: 'results_df' matrix not found. Please execute the optimization sweep cell first.")